In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
import os

# potential field based deep metric learning


def generate_static_pfml_data(n_samples=1500, seed=42, save_dir='./data'):
    centers = [
        [-5, -5, -5], [-4, -8, -4],  # Sub-clusters for Class 0
        [ 5,  5,  5], [ 8,  4,  6],  # Sub-clusters for Class 1
        [-5,  5, -5], [ 5, -5,  5]   # Sub-clusters for Class 2 (Widely separated)
    ]
    
    X, y_blobs = make_blobs(
        n_samples=n_samples,
        centers=centers,
        cluster_std=,
        random_state=seed
    )

    y = np.zeros_like(y_blobs)
    y[np.isin(y_blobs, [0, 1])] = 0 # class 0 
    y[np.isin(y_blobs, [2, 3])] = 1 # class 1
    y[np.isin(y_blobs, [4, 5])] = 2 # class 3
    
    os.makedirs(save_dir, exist_ok=True)
    x_path = os.path.join(save_dir, 'pfml_X_static.csv')
    y_path = os.path.join(save_dir, 'pfml_y_static.csv')
    
    np.save(x_path, X)
    np.save(y_path, y)
    
    print(f'Dataset saved to {save_dir}')
    print(f'X shape: {X.shape}, y shape: {y.shape}')
    
    return X, y

X, y = generate_static_pfml_data()



Dataset saved to ./data
X shape: (1500, 3), y shape: (1500,)


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from PIL import Image
import io
import os

# ==========================================
# 1. The PFML Loss Module
# ==========================================
class PFMLLoss(nn.Module):
    def __init__(self, num_classes, proxies_per_class, embedding_dim):
        super(PFMLLoss, self).__init__()
        self.num_classes = num_classes
        self.proxies_per_class = proxies_per_class
        
        # Initialize Proxies randomly
        total_proxies = num_classes * proxies_per_class
        self.proxies = nn.Parameter(torch.randn(total_proxies, embedding_dim) * 2)
        self.proxy_labels = torch.arange(num_classes).repeat_interleave(proxies_per_class)

    def forward(self, embeddings, labels):
        device = embeddings.device
        self.proxy_labels = self.proxy_labels.to(device)
        
        # Pairwise Distances
        dist_matrix = torch.cdist(embeddings, self.proxies, p=2) ** 2 
        
        # Masks
        labels_expanded = labels.unsqueeze(1)
        proxy_labels_expanded = self.proxy_labels.unsqueeze(0)
        pos_mask = (labels_expanded == proxy_labels_expanded)
        neg_mask = (labels_expanded != proxy_labels_expanded)
        
        # Attraction to nearest positive proxy
        pos_dists = dist_matrix.masked_fill(~pos_mask, float('inf'))
        min_pos_dists, _ = torch.min(pos_dists, dim=1)
        loss_att = min_pos_dists.mean() 
        
        # Repulsion from all negative proxies (margin = 5.0 for 2D visual scale)
        margin = 5.0
        neg_dists = dist_matrix.masked_fill(~neg_mask, float('inf'))
        repulsive_force = torch.clamp(margin - torch.sqrt(neg_dists), min=0.0)
        loss_rep = repulsive_force.mean()
        
        return loss_att + loss_rep

# ==========================================
# 2. Data Generation (2D Toy Data)
# ==========================================
def get_toy_data():
    centers = [[-5, -5], [-4, -8], [5, 5], [8, 4], [-5, 5], [5, -5]]
    X, y_blobs = make_blobs(n_samples=300, centers=centers, cluster_std=5.0, random_state=42)
    y = np.zeros_like(y_blobs)
    y[np.isin(y_blobs, [0, 1])] = 0
    y[np.isin(y_blobs, [2, 3])] = 1
    y[np.isin(y_blobs, [4, 5])] = 2
    return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long)

# ==========================================
# 3. Training & GIF Generation Loop
# ==========================================
def train_and_animate():
    print("Initializing environment...")
    X_init, y = get_toy_data()
    
    # We make the embeddings learnable so we can watch them move!
    embeddings = nn.Parameter(X_init.clone())
    
    # 3 Classes, 2 Proxies per class, 2D space
    loss_fn = PFMLLoss(num_classes=3, proxies_per_class=2, embedding_dim=2)
    
    # Optimizer handles both the points AND the proxies
    optimizer = torch.optim.Adam([embeddings, loss_fn.proxies], lr=0.15)
    
    frames = []
    epochs = 150
    
    # Colors for our 3 classes
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
    
    print("Starting optimization loop...")
    for epoch in range(epochs):
        optimizer.zero_grad()
        loss = loss_fn(embeddings, y)
        loss.backward()
        optimizer.step()
        
        # Capture a frame every 2 epochs to keep the GIF smooth but not massive
        if epoch % 2 == 0:
            fig, ax = plt.subplots(figsize=(8, 8))
            ax.set_xlim(-10, 10)
            ax.set_ylim(-10, 10)
            ax.set_title(f"PFML Latent Space Optimization - Epoch {epoch}")
            
            # Detach tensors for plotting
            emb_np = embeddings.detach().numpy()
            prox_np = loss_fn.proxies.detach().numpy()
            prox_labels = loss_fn.proxy_labels.numpy()
            
            # Plot Embeddings
            for c in range(3):
                mask = (y.numpy() == c)
                ax.scatter(emb_np[mask, 0], emb_np[mask, 1], c=colors[c], alpha=0.3, s=20)
                
            # Plot Proxies (Stars with black edges)
            for c in range(3):
                mask = (prox_labels == c)
                ax.scatter(prox_np[mask, 0], prox_np[mask, 1], c=colors[c], 
                           marker='*', s=300, edgecolor='black', linewidth=1.5, label=f'Class {c} Proxies')
            
            ax.legend(loc='upper right')
            
            # Save frame to memory buffer
            buf = io.BytesIO()
            plt.savefig(buf, format='png', bbox_inches='tight')
            buf.seek(0)
            frames.append(Image.open(buf))
            plt.close(fig)

    # Save to disk
    save_path = "pfml_training.gif"
    frames[0].save(save_path, format='GIF', append_images=frames[1:], 
                   save_all=True, duration=100, loop=0)
    
    print(f"Done! GIF saved successfully to {save_path}")

if __name__ == "__main__":
    train_and_animate()

Initializing environment...
Starting optimization loop...
Done! GIF saved successfully to pfml_training.gif
